### Install Automatic1111 Web UI
- First, clone the official Web UI repository into Kaggle’s working directory.

In [1]:
!git clone https://github.com/AUTOMATIC1111/stable-diffusion-webui /kaggle/working/stable-diffusion-webui
%cd /kaggle/working/stable-diffusion-webui
!git switch dev
!git pull

Cloning into '/kaggle/working/stable-diffusion-webui'...
remote: Enumerating objects: 34985, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 34985 (delta 0), reused 0 (delta 0), pack-reused 34982 (from 2)
Receiving objects: 100% (34985/34985), 35.66 MiB | 37.11 MiB/s, done.
Resolving deltas: 100% (24367/24367), done.
/kaggle/working/stable-diffusion-webui
Branch 'dev' set up to track remote branch 'dev' from 'origin'.
Switched to a new branch 'dev'
Already up to date.


### Download Stable Diffusion 1.5 Base Model
- Fetch the v1.5 checkpoint and a VAE (Variational Autoencoder) for better image quality.

In [2]:
!mkdir -p /kaggle/temp/models
!wget -O models/Stable-diffusion/v1-5-pruned-emaonly.safetensors "https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.safetensors"
!wget -O models/VAE/vae-ft-mse-840000-ema-pruned.safetensors "https://huggingface.co/stabilityai/sd-vae-ft-mse-original/resolve/main/vae-ft-mse-840000-ema-pruned.safetensors"

--2026-02-24 14:50:03--  https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.safetensors
Resolving huggingface.co (huggingface.co)... 13.226.251.112, 13.226.251.66, 13.226.251.20, ...
Connecting to huggingface.co (huggingface.co)|13.226.251.112|:443... connected.
HTTP request sent, awaiting response... 307 Temporary Redirect
Location: /stable-diffusion-v1-5/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.safetensors [following]
--2026-02-24 14:50:03--  https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.safetensors
Reusing existing connection to huggingface.co:443.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/66d19580e2632490a6bc5829/2ac63bfb6186057d88b65c3aa47ec90fd8d9aa3269164fc37fed1cb6f1a1efd0?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27v1-5-pruned-emaonly.safetensors%3B+filename%3D%22v1-5-pruned-emaonly.safetensors%22%3B&

### Install Dependencies
- Uninstall any pre-installed Torch versions and install the exact ones needed for our GPU setup, along with Web UI requirements and ngrok.


In [3]:
!pip uninstall -y torch torchvision torchaudio xformers peft accelerate diffusers transformers
!pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124
!pip install xformers==0.0.29.post3 --index-url https://download.pytorch.org/whl/cu124
!pip install -r /kaggle/working/stable-diffusion-webui/requirements_versions.txt
!pip install -qq pyngrok

Found existing installation: torch 2.6.0+cu124
Uninstalling torch-2.6.0+cu124:
  Successfully uninstalled torch-2.6.0+cu124
Found existing installation: torchvision 0.21.0+cu124
Uninstalling torchvision-0.21.0+cu124:
  Successfully uninstalled torchvision-0.21.0+cu124
Found existing installation: torchaudio 2.6.0+cu124
Uninstalling torchaudio-2.6.0+cu124:
  Successfully uninstalled torchaudio-2.6.0+cu124
Found existing installation: peft 0.16.0
Uninstalling peft-0.16.0:
  Successfully uninstalled peft-0.16.0
Found existing installation: accelerate 1.9.0
Uninstalling accelerate-1.9.0:
  Successfully uninstalled accelerate-1.9.0
Found existing installation: diffusers 0.34.0
Uninstalling diffusers-0.34.0:
  Successfully uninstalled diffusers-0.34.0
Found existing installation: transformers 4.53.3
Uninstalling transformers-4.53.3:
  Successfully uninstalled transformers-4.53.3
Looking in indexes: https://download.pytorch.org/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6

### Restart Session
- Restarting ensures Kaggle reloads the correct GPU libraries before we launch the Web UI (Important for GPU initialization).


### Configure ngrok
- Ngrok creates a secure URL so we can access the Web UI running in Kaggle from our browser

In [7]:
from pyngrok import ngrok

# Set your ngrok auth token (get it from ngrok.com)
ngrok.set_auth_token("")

### Move into the directory
- As we restarted the session, we need to move to the web-ui folder to start the UI


In [8]:
%cd /kaggle/working/stable-diffusion-webui

/kaggle/working/stable-diffusion-webui


### Launch the Web UI
- Finally, we run the launch command to start Stable Diffusion Web UI and open it through ngrok

In [ ]:
import threading
import time
import subprocess
import requests # New import
from pyngrok import ngrok

def kill_existing_tunnels():
    """Kill all existing ngrok tunnels to free up slots."""
    try:
        ngrok.disconnect_all()
        print("All existing ngrok tunnels disconnected.")
    except Exception as e:
        print(f"Error disconnecting tunnels: {e}")

def run_webui():
    """Function to run the Automatic1111 WebUI."""
    # Using --no-share to avoid Gradio's own public URL, as we are using ngrok.
    # The --listen flag is crucial for allowing external connections via ngrok.
    subprocess.run([
        "python", "launch.py",
        "--listen",
        "--port", "7860",
        "--enable-insecure-extension-access",
        "--no-download-sd-model",
        "--xformers",
        "--skip-torch-cuda-test",
        "--clip-models-path", "/kaggle/working/clip-vit-large-patch14"
    ])

def wait_for_server(port, timeout=120):
    """Waits for the server to start listening on the specified port."""
    start_time = time.time()
    while True:
        if time.time() - start_time > timeout:
            raise TimeoutError("Server did not start within the specified timeout.")
        try:
            requests.get(f"http://localhost:{port}")
            print(f"Server is up and running on port {port}.")
            return True
        except requests.exceptions.ConnectionError:
            print("Waiting for WebUI server to start...")
            time.sleep(20)

# --- Main execution flow ---

# 1. Clean up existing tunnels to avoid conflicts.
print("Cleaning up existing ngrok tunnels...")
kill_existing_tunnels()

# 2. Start the WebUI in a background thread.
print("Starting Automatic1111 WebUI...")
webui_thread = threading.Thread(target=run_webui, daemon=True)
webui_thread.start()

# 3. Wait for the WebUI server to become available.
try:
    wait_for_server(7860)
except TimeoutError as e:
    print(f"Error: {e}")
    # You may want to stop the script here if the server fails to start.
    exit(1)

# 4. Create and display the ngrok tunnel.
print("Creating ngrok tunnel...")
try:
    public_url = ngrok.connect(7860)
    print(f"\n🎉 Stable Diffusion WebUI is accessible at:")
    print(f"🔗 {public_url}")
    print("\nClick the link above to access the WebUI interface!")

    # Keep the main thread alive so the WebUI thread continues to run
    # until the notebook cell execution is stopped.
    webui_thread.join()

except Exception as e:
    print(f"Error creating tunnel: {e}")
    print("Check your ngrok authentication token or try restarting the session.")

Cleaning up existing ngrok tunnels...
Error disconnecting tunnels: module 'pyngrok.ngrok' has no attribute 'disconnect_all'
Starting Automatic1111 WebUI...
Waiting for WebUI server to start...


Cloning into '/kaggle/working/stable-diffusion-webui/repositories/stable-diffusion-webui-assets'...
Cloning into '/kaggle/working/stable-diffusion-webui/repositories/stable-diffusion-stability-ai'...
Cloning into '/kaggle/working/stable-diffusion-webui/repositories/generative-models'...
Cloning into '/kaggle/working/stable-diffusion-webui/repositories/k-diffusion'...
Cloning into '/kaggle/working/stable-diffusion-webui/repositories/BLIP'...


Waiting for WebUI server to start...


2026-02-24 15:00:25.835327: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771945226.034523     241 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771945226.091936     241 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
/usr/local/lib/python3.11/dist-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Waiting for WebUI server to start...
Python 3.11.13 (main, Jun  4 2025, 08:57:29) [GCC 11.4.0]
Version: v1.10.1-93-gfd68e0c3
Commit hash: fd68e0c3846b07c637c3d57b0c38f06c8485a753
Installing clip
Cloning assets into /kaggle/working/stable-diffusion-webui/repositories/stable-diffusion-webui-assets...
Cloning Stable Diffusion into /kaggle/working/stable-diffusion-webui/repositories/stable-diffusion-stability-ai...
Cloning Stable Diffusion XL into /kaggle/working/stable-diffusion-webui/repositories/generative-models...
Cloning K-diffusion into /kaggle/working/stable-diffusion-webui/repositories/k-diffusion...
Cloning BLIP into /kaggle/working/stable-diffusion-webui/repositories/BLIP...
Launching Web UI with arguments: --listen --port 7860 --enable-insecure-extension-access --no-download-sd-model --xformers --skip-torch-cuda-test --clip-models-path /kaggle/working/clip-vit-large-patch14
Calculating sha256 for /kaggle/working/stable-diffusion-webui/models/Stable-diffusion/v1-5-pruned-emaonly

/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Server is up and running on port 7860.
Creating ngrok tunnel...

🎉 Stable Diffusion WebUI is accessible at:
🔗 NgrokTunnel: "https://nasological-incompressibly-georgiana.ngrok-free.dev" -> "http://localhost:7860"

Click the link above to access the WebUI interface!
